# 02 — Error analysis of the one-shot test evaluation

The test sets were scored **once** by `make evaluate` (V4). This notebook only reads the saved predictions and metrics; it never re-scores or re-tunes anything on test. The model is loaded solely to draw Grad-CAM heatmaps. Run with `make error-analysis`.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from defect_detection.config import load_config
from defect_detection.core.model_meta import ModelMeta
from defect_detection.training import error_analysis as ea
from defect_detection.training.trainer import load_trained_model, resolve_device

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
config = load_config(Path(os.environ.get("DD_CONFIG", "configs/train.yaml")))
RUNS = config.train.output_dir
RUN = RUNS / (RUNS / "LATEST").read_text(encoding="utf-8").strip()
RAW, FIGURES = config.data.raw_dir, Path("reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)
metrics = json.loads((RUN / "test_metrics.json").read_text(encoding="utf-8"))
meta = ModelMeta.model_validate_json(
    (RUN / "model_meta.candidate.json").read_text(encoding="utf-8")
)
predictions = pd.read_csv(RUN / "test_predictions.csv")
trained = load_trained_model(RUN, resolve_device("cpu"))
BAND = (meta.review_band.low, meta.review_band.high)
print(RUN.name, "T =", meta.temperature, "threshold =", meta.threshold, "band =", BAND)

## 1. Headline metrics (operating threshold, 95% bootstrap CIs)

In [ ]:
rows = []
for split, r in metrics["results"].items():
    for name, ci in r["at_operating_threshold"].items():
        if name != "_meta":
            rows.append(
                {
                    "split": split,
                    "metric": name,
                    "point": ci["point"],
                    "ci_low": ci["low"],
                    "ci_high": ci["high"],
                }
            )
pd.DataFrame(rows).pivot(index="metric", columns="split").round(4)

In [ ]:
display(Image(str(RUN / "test_confusion.png")))
display(Image(str(RUN / "test_curves.png")))

## 2. Every error

In [ ]:
errors = ea.error_table(predictions)
print(errors.groupby(["split", "error_type"]).size())
errors

## 3. Missed defects (false negatives) with Grad-CAM — the costly errors

In [ ]:
for split in ("test", "external_test"):
    fn = errors[(errors["split"] == split) & errors["error_type"].str.startswith("FN")]
    rows = predictions.set_index("rel_path").loc[fn["rel_path"]].reset_index().head(24)
    if len(rows):
        fig = ea.gallery(rows, RAW, trained.model, trained.spec, f"{split}: missed defects")
        fig.savefig(FIGURES / f"errors_fn_{split}.png", dpi=90)

## 4. False alarms (false positives) with Grad-CAM

In [ ]:
for split in ("test", "external_test"):
    fp = errors[(errors["split"] == split) & errors["error_type"].str.startswith("FP")]
    rows = predictions.set_index("rel_path").loc[fp["rel_path"]].reset_index().head(24)
    if len(rows):
        fig = ea.gallery(rows, RAW, trained.model, trained.spec, f"{split}: false alarms")
        fig.savefig(FIGURES / f"errors_fp_{split}.png", dpi=90)

## 5. Correct predictions with Grad-CAM (does the model look at the defect?)

In [ ]:
sample = ea.sample_correct(predictions, per_group=4, seed=config.seed)
fig = ea.gallery(
    sample, RAW, trained.model, trained.spec, "correct predictions (4 per split and class)"
)
fig.savefig(FIGURES / "gradcam_correct.png", dpi=90)

## 6. Confidence and the review band

In [ ]:
fig = ea.confidence_histograms(predictions, meta.threshold, BAND)
fig.savefig(FIGURES / "confidence_histograms.png", dpi=110)
pd.DataFrame({s: r["review_band"] for s, r in metrics["results"].items()})

## 7. Performance by similarity to the nearest TRAIN image

High-similarity buckets contain near-copies of training parts; the low-similarity buckets show how the model does on unfamiliar parts.

In [ ]:
pd.concat(
    {
        s: pd.DataFrame(r["by_similarity_to_train"]).set_index("similarity")
        for s, r in metrics["results"].items()
    }
)

## 8. Lighting-shortcut probe

EDA found brightness alone separates the classes on train/val (AUC 0.883). Within a true class, P(defective) should not track brightness.

In [ ]:
for split, r in metrics["results"].items():
    probe = r["brightness_probe"]
    print(split, {k: round(v, 3) for k, v in probe.items() if k.startswith("spearman")})
    display(pd.DataFrame(probe["errors_by_brightness_quartile"]))

## 9. Classifier vs PatchCore anomaly baseline

In [ ]:
anomaly_path = Path("reports/anomaly_baseline.json")
if anomaly_path.exists():
    anomaly = json.loads(anomaly_path.read_text(encoding="utf-8"))
    table = {}
    for split in ("test", "external_test"):
        cls = metrics["results"][split]["at_operating_threshold"]
        table[("classifier", split)] = {
            m: cls[m]["point"] for m in ("roc_auc", "pr_auc", "precision", "recall", "f1")
        }
        table[("patchcore", split)] = {
            m: anomaly[split][m] for m in ("roc_auc", "pr_auc", "precision", "recall", "f1")
        }
    display(pd.DataFrame(table).T.round(4))
    print("PatchCore fit:", anomaly["fit"])
    print(
        "PatchCore ms/img:", {s: anomaly[s]["ms_per_image_cpu"] for s in ("test", "external_test")}
    )

## 10. Findings

Written after reviewing the outputs above (see the V4 gate report).